# Combined EDA — German grid load and residual load

**Spec:** [`.claude/specs/03-combined-cherry-picked-eda.md`](../../.claude/specs/03-combined-cherry-picked-eda.md)
(sections `03.1`–`03.7`) · **Data:** `data/smard.csv` (SMARD / Bundesnetzagentur, hourly, region DE)

## What this notebook is

The team's four individual explorations — [`EDA-hari.ipynb`](EDA-hari.ipynb),
[`EDA-magc.ipynb`](EDA-magc.ipynb), [`EDA-robert.ipynb`](EDA-robert.ipynb) and
[`EDA-simple-claude.ipynb`](EDA-simple-claude.ipynb) — consolidated into one notebook that runs
top to bottom against `data/smard.csv` alone. Only plot cells **tagged** in a source notebook are
included, every "duplicate" and "combine with X" tag is resolved into a decision recorded in the
relevant sub-spec, and everything is rebuilt against one shared frame, naming convention and unit
system rather than four different ones. The four source notebooks are not modified.

## Sections

| # | Sub-spec | Section |
|---|---|---|
| 1 | [`03.1`](../../.claude/specs/03.1-setup.md) | Setup — loading, helpers, `YEARS`, `LOADED` |
| 2 | [`03.2`](../../.claude/specs/03.2-sanity-check.md) | Sanity check |
| 3 | [`03.3`](../../.claude/specs/03.3-univariate-and-time-structure.md) | Univariate and time structure |
| 4 | [`03.4`](../../.claude/specs/03.4-two-series-comparison-views.md) | Two-series comparison views |
| 5 | [`03.5`](../../.claude/specs/03.5-calendar-structure-heatmaps.md) | Calendar structure and single events |
| 6 | [`03.6`](../../.claude/specs/03.6-temporal-dependence-and-extremes.md) | Temporal dependence, decomposition, ramps and extremes |
| 7 | [`03.7`](../../.claude/specs/03.7-context-appendix.md) | Context appendix, findings, self-check |

## Conventions

- The shared hourly frame is **`time_series`**, never `ts`. `SERIES` names the eight data
  columns, `DERIVED` the ten calendar/helper columns — the split keeps `.corr()` and
  `.describe()` from treating `year` or `dow` as measurements.
- **No literal calendar year in code.** The record's extent is a snapshot, not a constant (the
  team intends to widen the fetch back to 2019), so year lists, anchor years and colour maps are
  derived from `YEARS` at run time. The one deliberate exception is §5's Euro 2024 fixture dates,
  which are historical fact rather than a property of the record.
- **No thresholds.** This notebook defines no risk flag, cut-off or labelled column. Where it
  shows "the tail hours" it selects them by rank, for description only.
- **One colour configuration.** §1.2 assigns a colour to every feature once, so the same series
  is the same colour in every figure of this notebook. No plot picks its own colours.
- **Descriptive slices stay local.** Rank- or date-based selections are computed inside their own
  plotting cell and never persisted onto `time_series`; §7's self-check proves mechanically that
  no flag column crept in.

---

## 1 — Setup

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it by
running [`notebooks/API-connection.ipynb`](../API-connection.ipynb) top to bottom — it pulls the
SMARD API (no key required) and writes the file in German Excel CSV format
(`sep=";"`, `decimal=","`, `utf-8-sig`).

The data directory is resolved by walking **upward** from the working directory rather than by a
fixed `"../../data/smard.csv"`: this notebook sits two levels below the repo root, and a
hardcoded depth breaks as soon as the kernel starts somewhere else (nbconvert from the root, for
instance).

In [ ]:
from pathlib import Path

import holidays
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams["figure.max_open_warning"] = 0  # this notebook draws ~30 figures on purpose

# Walk up from the working directory to the first parent holding a `data/` folder, so the
# notebook runs unmodified from notebooks/01_eda/ (Jupyter) or from the repo root (nbconvert).
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(f"pandas {pd.__version__} · numpy {np.__version__} · seaborn {sns.__version__}")
print(f"reading {DATA}")

### 1.1 — Helpers

Copied unchanged in behaviour from [`EDA-simple-claude.ipynb`](EDA-simple-claude.ipynb), which
holds the project's canonical copies (originally from [`EDA-robert.ipynb`](EDA-robert.ipynb),
corrected there). They are defined **once, here**, and reused — not redefined — by every later
section.

- `_complete_periods` holds the edge rule in exactly one place, shared by both aggregation
  helpers. The rule is *drop periods the data does not fully cover*, which is not the same as
  "drop the first and the last": this record starts on a month boundary, so its opening month is
  complete and only the trailing one goes. A plain `.resample()` produces fake edge dips.
- `style_timeseries` and `seasonal_plot` both require `ylabel`, and `seasonal_plot` actually
  applies it (the original hard-coded `"MWh"` and discarded the argument).

In [ ]:
def _complete_periods(index, freq):
    """The calendar periods of `freq` that `index` covers completely.

    The one edge rule of this notebook, in one place. A period counts only if it starts no
    earlier than the first observation and ends no later than the last observation's closing
    edge. Both `period_mean` and `period_energy` defer to this.
    """
    periods = index.to_period(freq).unique().sort_values()
    complete = (periods.start_time >= index.min()) & (
        periods.end_time <= index.max() + pd.Timedelta("1h")
    )
    return periods[complete]


def period_mean(series, freq):
    """Mean of `series` per calendar period (`"W"`, `"M"`, ...), indexed by period start.

    Periods that the data does not cover completely are dropped, so the edges of a plot are not
    partial-period artefacts. Note the rule is "not fully covered", not "the first and the last".
    """
    agg = series.groupby(series.index.to_period(freq)).mean()
    agg = agg.loc[_complete_periods(series.index, freq)]
    agg.index = agg.index.start_time
    return agg


def period_energy(series, freq, drop_incomplete=True):
    """Per-period aggregate of `series` in both project reporting units.

    Returns a DataFrame indexed by period start:

    ``mwh_per_day``
        period sum / calendar days in the period — the energy view (MWh/day).
    ``avg_mw``
        period sum / hours **actually present** — the level view (MW). Deliberately not
        ``mwh_per_day / 24``: a month containing the spring DST switch holds 743 hours, not 744.
    ``hours``, ``days``
        the two denominators, exposed so a comparison table needs no second copy of this
        arithmetic.

    Incomplete periods are dropped by the same `_complete_periods` rule as `period_mean`.
    """
    grouped = series.groupby(series.index.to_period(freq))
    total, hours = grouped.sum(), grouped.size()
    periods = total.index

    if drop_incomplete:
        keep = _complete_periods(series.index, freq)
        total, hours, periods = total.loc[keep], hours.loc[keep], keep

    # Freq-generic: 7 for every week, 28-31 for months. `days_in_month` would be "M"-only.
    days = (periods.end_time.normalize() - periods.start_time).days + 1

    # .to_numpy() on every right-hand side: aligning a PeriodIndex-backed Series against a
    # DatetimeIndex-derived array silently yields all-NaN.
    return pd.DataFrame(
        {
            "mwh_per_day": total.to_numpy() / days,
            "avg_mw": total.to_numpy() / hours.to_numpy(),
            "hours": hours.to_numpy(),
            "days": np.asarray(days),
        },
        index=periods.start_time,
    )

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required: every plot must state whether it shows MWh, average MW or MWh/day.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")


def seasonal_plot(df, y_value, title, ylabel):
    """Creates a seasonal plot from a dataframe.

    Args:
        df (DataFrame): frame with separate `month` and `year` columns, already aggregated to
            one row per (year, month). Passing raw hourly data makes seaborn bootstrap a
            confidence interval per cell over tens of thousands of rows — minutes of runtime,
            meaningless band.
        y_value (str): name of the y-value to plot
        title (str): title of the plot
        ylabel (str): axis description, including units. Required, and actually applied.
    """
    fig, ax = plt.subplots(figsize=(14, 5))

    sns.lineplot(
        data=df,
        x="month",
        y=y_value,
        hue="year",
        palette="viridis",
        legend=True,
        ax=ax
    )
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(range(1, 13))
    plt.tight_layout()
    plt.show()

### 1.2 — Colour configuration

**The one place colours are decided.** Every figure in this notebook reads its colours from the
mappings below, so a series looks the same wherever it appears and no plot invents its own
palette.

The base palette is [`EDA-hari.ipynb`](EDA-hari.ipynb)'s, with two team decisions applied on top:
**`grid_load` is red** and **`residual_load` near-black** — the two headline series, kept maximally
distinct from each other and from the generation series. Each SMARD day-ahead forecast takes its
measured counterpart's colour and is drawn **dashed**, so a forecast and its actual read as the
same quantity rather than as two unrelated lines.

Where a plot splits by **day type** rather than by series, weekdays are orange and weekends blue —
warm for working days, cool for the weekend, and both kept clear of `grid_load`'s red so the split
never reads as a change of series. Where it splits by **residual-load extreme**, the low
(oversupply) end is blue and the high (tight-margin) end red.

Year-coded plots call `year_colors(...)`, which samples a continuous colormap over whatever years
the record happens to contain — no fixed year-to-colour table, which would break the moment the
fetch range widens.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# Base palette, from EDA-hari.ipynb.
COLORS = {
    "navy":   "#17365D",
    "blue":   "#2C6EBA",
    "teal":   "#4AA3A5",
    "green":  "#2F8F5B",
    "mint":   "#6FBF9B",
    "gold":   "#D9A53A",
    "orange": "#E95D0F",
    "coral":  "#E76F51",
    "red":    "#B10F0F",
    "purple": "#9274B8",
    "gray":   "#AEB8C5",
    "black":  "#1C1C1C",
    "ink":    "#17365D",
    "muted":  "#707B8C",
    "grid":   "#E3E8EF",
    "cream":  "#F7F3E7",
}

# One colour per feature. Forecasts deliberately share their measured counterpart's colour.
SERIES_COLOR = {
    "grid_load":         COLORS["red"],
    "residual_load":     COLORS["black"],
    "wind_on":           COLORS["teal"],
    "wind_off":          COLORS["blue"],
    "solar":             COLORS["gold"],
    "renewables":        COLORS["green"],
    "wind_total":        COLORS["purple"],
    "fc_grid_load":      COLORS["red"],
    "fc_res":            COLORS["black"],
    "fc_gen_wind_solar": COLORS["green"],
}

SERIES_LABEL = {
    "grid_load":         "Grid load",
    "residual_load":     "Residual load",
    "wind_on":           "Onshore wind",
    "wind_off":          "Offshore wind",
    "solar":             "Solar",
    "renewables":        "Wind + solar",
    "wind_total":        "Wind (on + offshore)",
    "fc_grid_load":      "Forecast grid load",
    "fc_res":            "Forecast residual load",
    "fc_gen_wind_solar": "Forecast wind + solar",
}

# Day type is a dimension of its own, not a series: warm for working days, cool for weekends.
# Kept distinct from grid_load's red so a day-type split never reads as a different series.
DAY_TYPE_COLOR = {"weekday": COLORS["orange"], "weekend": COLORS["blue"]}

# The two residual-load extremes, wherever they are shown as a pair: cool for the oversupply end,
# hot for the tight-margin end.
TAIL_COLOR = {"low": COLORS["blue"], "high": COLORS["red"]}

FORECAST_COLS = ["fc_gen_wind_solar", "fc_grid_load", "fc_res"]
HEADLINE_COLS = ["grid_load", "residual_load"]  # drawn slightly heavier than the rest


def series_style(col, **overrides):
    """Line kwargs for `col`: colour, width and dash pattern, from the configuration above."""
    style = {
        "color": SERIES_COLOR[col],
        "linewidth": 2.4 if col in HEADLINE_COLS or col in FORECAST_COLS[1:] else 1.8,
        "linestyle": "--" if col in FORECAST_COLS else "-",
        "label": SERIES_LABEL[col],
    }
    style.update(overrides)
    return style


def year_colors(years, cmap="viridis"):
    """A colour per year, sampled from `cmap` — never a fixed year-to-colour table."""
    years = list(years)
    sampled = plt.get_cmap(cmap)(np.linspace(0.05, 0.95, len(years)))
    return dict(zip(years, sampled))


# Heatmap scales. `grid_load` is strictly positive and takes a sequential ramp towards its own
# red; `residual_load` crosses zero and needs a diverging, zero-centred scale — they cannot share
# one colour bar.
GRID_LOAD_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_grid_load",
    ["#FBF1EF", "#F0C3B8", "#DE8B76", COLORS["coral"], COLORS["red"]],
)
RESIDUAL_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_residual",
    [COLORS["green"], "#B9DCC7", "#FFFFFF", "#8A8A8A", COLORS["black"]],
)
# Hari's remaining scales, reused unchanged by the plots ported from that notebook.
SUPPORT_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_support",
    ["#F7F9FC", "#D7E6F3", "#82B6D9", COLORS["blue"], COLORS["navy"]],
)
DIV_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_diverging",
    [COLORS["teal"], "#DCEEEF", "#FFFFFF", "#F8D1C5", COLORS["coral"]],
)
WIND_SOLAR_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_wind_solar",
    ["#F8F3E7", "#F3C879", "#9BD0C3", COLORS["teal"], COLORS["green"]],
)

print(f"{len(SERIES_COLOR)} features coloured, {len(COLORS)} base colours")

### 1.3 — Load and prepare

The CSV is German Excel format, so every numeric column arrives as text with a comma decimal
separator. The failure mode to guard against is silent: unconverted columns land as a string
dtype, every aggregate still computes something, and every number is wrong. The dtype assertion
below is the guard — a positive `is_float_dtype` test rather than `!= object`, because under
pandas 3 an unconverted column lands as `StringDtype`, which `!= object` would wave straight
through.

The flat, `RangeIndex`ed frame is called `raw` and is **deleted** at the end of the loading
cells. Everything downstream uses `time_series`, so the two cannot drift apart.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them. Note the inconsistent
# capitalisation in the source ("Grid Load" vs "Forecast Grid load") — reproduced deliberately.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid load": "fc_grid_load",
    "Forecast Residual Load": "fc_res",
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

print("as read from disk — note the comma decimals and the string dtypes:")
display(raw.head(3))
display(raw.dtypes.to_frame("dtype"))

In [ ]:
raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cells

time_series.head(3)

In [ ]:
print(f"shape           : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"index           : {time_series.index.min()}  ->  {time_series.index.max()}")
print(
    f"index monotonic : {time_series.index.is_monotonic_increasing}, "
    f"unique: {time_series.index.is_unique}"
)
display(time_series.dtypes.to_frame("dtype"))

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

# Snapshot for §7's closing self-check, taken before any other cell can touch the frame: a cell
# inserted anywhere in between that mutates `time_series` is caught regardless of which vintage
# of the CSV was loaded.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}
print(f"\nLOADED = {LOADED}")

display(time_series.describe().T)

### 1.4 — Derived columns, `SERIES` / `DERIVED`, and `YEARS`

All derived columns are defined **here, in one place**, immediately after loading. Two of them
are needed by the sanity check itself (`hour` for the night-solar test, `renewables` as the
wind+solar aggregate), so they cannot wait for the section that first plots them.

`YEARS` is computed from the loaded data and is the only permitted source of year information in
the rest of the notebook — no section may write a calendar year into code (see the note on the
one deliberate exception at the top).

`spans_gap` marks the row *following* a gap in the hourly index. The record's only gaps are the
spring DST switches, where the local hour 02:00 does not exist; `.diff()`, `.shift()` and rolling
windows all need that flag, so it is created here with everything else.

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_res",
]

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)
# True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")
time_series[DERIVED].head(3)

### 1.5 — Initial self-check

Structural only, and deliberately free of any hardcoded row count or date bound: the record's
extent is expected to change. §7 re-runs the same invariants at the end of the notebook, plus a
comparison against `LOADED`, which together prove that nothing in between mutated the frame.

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique")
print(f"  columns == SERIES + DERIVED ({len(SERIES) + len(DERIVED)} columns)")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")

---

## 2 — Sanity check

A deliberately **small** slice of [`01-Simple-EDA.md`](../../.claude/specs/01-Simple-EDA.md)'s
data-quality audit, reproduced here so this notebook does not depend on
[`EDA-simple-claude.ipynb`](EDA-simple-claude.ipynb) having been run first — two checks, not the
whole audit — followed by a one-week plausibility plot.

Three data-quality views from the source notebooks are deliberately **absent**: the residual-load
identity check, the raw post-load lineplot and the styled DST-gap timeline. Their source cells
carry no tag, and "no tag, no plot" is applied uniformly. The one-week plot below is the mirror
case: its source cell is untagged too, so the *cell* is not ported — the team asked for the idea,
and it is rebuilt here against `time_series` in the project's own units.

### 2.1 — Missing values and duplicate timestamps

In [ ]:
missing = pd.DataFrame(
    {
        "n_missing": time_series[SERIES].isna().sum(),
        "share_%": (time_series[SERIES].isna().mean() * 100).round(3),
    }
)
display(missing)

n_duplicates = int(time_series.index.duplicated().sum())
print(f"missing values across all {len(SERIES)} series : {int(missing['n_missing'].sum())}")
print(f"duplicate timestamps                     : {n_duplicates}")

**Zero missing values and zero duplicate timestamps.** An explicit zero is a finding, not the
absence of one: no imputation strategy is needed for the columns themselves, so every `NaN` that
turns up later in this notebook is one *we* created — by a `.diff()`, a lag or an incomplete
aggregation period — rather than one that arrived with the data.

The zero duplicate count is worth reading carefully. The record does lose hours at each autumn
DST fold, where 02:00 occurs twice locally; SMARD collapses that repeated hour rather than
emitting it twice, so the fold produces neither a duplicate timestamp nor an index gap. Zero
duplicates therefore means *no hour appears twice*, not *the record is hour-complete*.

### 2.2 — Value ranges

A naive range check drowns in false positives, so the night-solar test carries an explicit
tolerance: night-time `solar` must be below **0.5 % of the series maximum**, not exactly zero.
Any real violation is listed by timestamp — reported, never filtered or corrected.

In [ ]:
NIGHT_HOURS = [22, 23, 0, 1, 2, 3]
solar_tol = 0.005 * time_series["solar"].max()
night = time_series["hour"].isin(NIGHT_HOURS)

checks = {
    "wind_off < 0": time_series["wind_off"] < 0,
    "wind_on < 0": time_series["wind_on"] < 0,
    "solar < 0": time_series["solar"] < 0,
    "grid_load <= 0": time_series["grid_load"] <= 0,
    f"night solar > {solar_tol:,.1f} MWh": night & (time_series["solar"] > solar_tol),
}

print(f"solar max {time_series['solar'].max():,.1f} MWh -> tolerance {solar_tol:,.1f} MWh")
print(f"night hours defined as {NIGHT_HOURS}\n")

for name, mask in checks.items():
    print(f"{name:<38} {int(mask.sum()):>6} violations")
    if mask.any():
        display(time_series.loc[mask, SERIES])  # list every real violation by timestamp

print(
    f"\nnight solar: max {time_series.loc[night, 'solar'].max():,.2f} MWh, "
    f"{int((time_series.loc[night, 'solar'] == 0).sum()):,} of {int(night.sum()):,} "
    "night hours are exactly zero"
)

### 2.3 — One ordinary week, end to end

Before summarising several years, look at a single seven-day window and ask whether the series
are physically plausible: load should keep a daily rhythm with a weekend dip, solar should switch
off at night, wind should wander freely, and `residual_load` should fall when wind and solar
support is strong.

The week is found **programmatically**: the first Monday 00:00 → Sunday 23:00 window with 168
consecutive hourly rows and no missing value in any of the eight series. It is not hand-picked,
and it is not assumed to be the record's first calendar week — the record can open mid-week, and
any week containing a spring DST switch is 167 rows and therefore disqualified.

In [ ]:
WEEK_HOURS = 24 * 7

valid = time_series[SERIES].notna().all(axis=1)
candidate_starts = time_series.index[(time_series["dow"] == 0) & (time_series["hour"] == 0)]

week_start, skipped = None, 0
for start in candidate_starts:
    expected = pd.date_range(start, periods=WEEK_HOURS, freq="h")
    # A week containing a spring DST switch is short one row, so `isin` also rules out gaps.
    if not expected.isin(time_series.index).all() or not valid.loc[expected].all():
        skipped += 1
        continue
    week_start = start
    break

if week_start is None:
    raise RuntimeError(
        "no fully valid Monday-to-Sunday week found — every candidate week has a gap or a NaN"
    )

week_index = pd.date_range(week_start, periods=WEEK_HOURS, freq="h")
week = time_series.loc[week_index]

print(f"first row in the record   : {time_series.index.min()} ({time_series.index.min():%A})")
print(f"candidate Mondays skipped : {skipped}")
print(f"week selected             : {week.index[0]:%a %Y-%m-%d %H:%M} -> {week.index[-1]:%a %Y-%m-%d %H:%M}")
print(f"rows                      : {len(week)}")

Both plots below cover exactly that week. Values are hourly, i.e. energy per hourly interval in
**MWh** — over an hour that number is also the average power in MW. Colours come from §1.2:
`grid_load` red, `residual_load` near-black, and each forecast dashed in its measured counterpart's
colour.

In [ ]:
def plot_week(frame, columns, title):
    """One week of hourly values, MWh. Returns the plotted span so §2.4 can compare the two."""
    fig, ax = plt.subplots(figsize=(13.5, 5.5))

    for col in columns:
        ax.plot(frame.index, frame[col].to_numpy(), **series_style(col))

    ax.axhline(0, color=COLORS["muted"], lw=0.8)
    ax.set_title(title, fontsize=15, pad=12)
    ax.set_ylabel("MWh", color="grey")
    ax.set_xlabel("Europe/Berlin time", labelpad=8)
    ax.grid(axis="y", color=COLORS["grid"], lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a\n%d %b"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
    ax.legend(ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.24))
    plt.tight_layout()
    plt.show()

    return frame.index[0], frame.index[-1]


PHYSICAL_COLS = ["grid_load", "residual_load", "wind_on", "wind_off", "solar"]
physical_span = plot_week(
    week,
    PHYSICAL_COLS,
    f"Measured series, {week.index[0]:%d %b %Y} – {week.index[-1]:%d %b %Y}",
)

In [ ]:
# FORECAST_COLS comes from §1.2 — each forecast keeps its measured counterpart's colour, dashed.
forecast_span = plot_week(
    week,
    FORECAST_COLS,
    f"SMARD day-ahead forecasts, same week ({week.index[0]:%d %b %Y} – {week.index[-1]:%d %b %Y})",
)

**The series behave as they physically should.** `grid_load` repeats a clear daily cycle with a
visibly lower weekend, `solar` returns to zero every night and peaks at midday, wind moves on its
own multi-day rhythm, and `residual_load` tracks `grid_load` minus the wind and solar support —
dipping hardest exactly where wind and solar are strongest. The forecast panel shows the same
shapes: the SMARD day-ahead series are close enough to the measured ones that their errors, not
their shapes, are the interesting quantity — which is why `fc_res` is the benchmark this project
has to beat.

### 2.4 — Self-check

In [ ]:
assert len(week) == WEEK_HOURS, len(week)
assert week.index[0].dayofweek == 0 and week.index[0].hour == 0, week.index[0]
assert week.index[-1].dayofweek == 6 and week.index[-1].hour == 23, week.index[-1]
assert (week.index.to_series().diff().dropna() == pd.Timedelta("1h")).all(), "internal gap"
assert week[SERIES].notna().all().all(), "NaN inside the selected week"
assert physical_span == forecast_span, (physical_span, forecast_span)
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

print("sanity-check self-check passed")
print(f"  {len(week)} consecutive rows, {week.index[0]:%A %H:%M} -> {week.index[-1]:%A %H:%M}")
print(f"  both plots cover {physical_span[0]:%Y-%m-%d} -> {physical_span[1]:%Y-%m-%d}")
print(f"  no column added to time_series ({len(time_series.columns)} columns)")

---

## 3 — Univariate and time structure

Eight views, all ported from [`EDA-simple-claude.ipynb`](EDA-simple-claude.ipynb)'s tagged cells:
what each series looks like on its own, and what calendar structure it carries. Nothing here
overlaps with the other three source notebooks, so no merge decisions were needed — only the
`time_series` rename, the shared helpers from §1, and the colour configuration from §1.2.

Six otherwise-central plots from the same source notebook are **absent**, their cells being
untagged: the residual-load identity histogram, the `SERIES` boxplot, the Jan–Sep matched trend
bars, the Pearson correlation heatmap, the residual-vs-renewables scatter and the negative-share
heatmap. That was a discussed cut, not an oversight — restoring any of them means tagging the
source cell and revising
[`03.3`](../../.claude/specs/03.3-univariate-and-time-structure.md).

### 3.1 — Every series over the full record

Weekly and monthly means per series, both via `period_mean`, so the incomplete opening/closing
periods are dropped rather than drawn as fake edge dips. Each panel carries its series' own colour
from §1.2.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 17))

for ax, col in zip(axes.flat, SERIES):
    weekly = period_mean(time_series[col], "W")
    monthly = period_mean(time_series[col], "M")
    ax.plot(
        weekly.index, weekly.to_numpy(),
        color=SERIES_COLOR[col], linewidth=1.0, alpha=0.45, label="weekly mean",
    )
    ax.plot(
        monthly.index, monthly.to_numpy(),
        color=SERIES_COLOR[col], linewidth=2.0, label="monthly mean",
    )
    ax.axhline(0, color="0.6", linewidth=0.8, zorder=0)
    style_timeseries(ax, SERIES_LABEL[col], "average MW")
    ax.set_xlabel("week / month (labelled by its start)", color="grey", fontsize=9)
    ax.legend(frameon=False, fontsize=9)

fig.suptitle(
    "All eight series, weekly and monthly means (incomplete edge periods dropped)",
    fontsize=16, y=0.997,
)
plt.tight_layout()
plt.show()

**`solar` is the one series with an unmistakable trend**: its summer peaks grow from roughly
11 GW to over 16 GW across the record. Wind, on- and offshore, shows no comparable climb — it is
dominated by weather variance, and a single windy or still week moves the level further than the
whole record's trend does. `grid_load` drifts gently down and is otherwise remarkably stable.

`residual_load` inherits all of it: a downward drift driven mostly by solar build-out, on top of
the largest short-term variance of any series here. The gap between the thin weekly line and the
thick monthly one is the point of this figure — for the wind and residual panels, monthly means
hide swings that matter for a day-ahead forecast.

### 3.2 — Distribution of each series

Hourly values, so each number is a level reading: over a single hour, `MWh` and `MW` are the same
number, and the axis says `MW` rather than restating it as an energy-per-hour. The red line marks
zero, which only `residual_load` and its forecast ever cross.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 13))

for ax, col in zip(axes.flat, SERIES):
    ax.hist(time_series[col], bins=80, color=SERIES_COLOR[col])
    ax.axvline(0, color=COLORS["red"], linewidth=1.0)
    ax.set_title(SERIES_LABEL[col], fontsize=12, pad=8)
    ax.set_xlabel("MW")
    ax.set_ylabel("hours", color="grey")
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

fig.suptitle("Hourly distribution of each series (red line = zero)", fontsize=16, y=0.999)
plt.tight_layout()
plt.show()

`grid_load` is close to symmetric around a high level; the generation series are strongly
right-skewed with a mass near zero (solar is zero every night by construction); `residual_load` is
the only series with meaningful weight **below zero**, which is the phenomenon this project is
about. Note also that no series is log-transformable: `residual_load` goes negative, and the
generation series sit on zero.

### 3.3 — Annual seasonality

Monthly means, month on the x-axis, one line per year. The last year in the record is partial, so
its line stops early — that is the data ending, not a collapse.

In [ ]:
# The record's own end month, so no calendar literal is needed in the title.
last_complete = period_mean(time_series["grid_load"], "M").index.max()

for col in ["grid_load", "residual_load", "renewables", "solar"]:
    monthly = period_mean(time_series[col], "M").to_frame(col)
    monthly["year"] = monthly.index.year
    monthly["month"] = monthly.index.month
    seasonal_plot(
        monthly,
        col,
        f"{SERIES_LABEL[col]}: monthly mean by month of year, coloured by year "
        f"(last full month {last_complete:%b %Y})",
        "average MW",
    )

`grid_load` traces the expected winter-high / summer-low curve, and the year lines sit almost on
top of each other — demand seasonality is stable. `solar` is the mirror image and just as regular.
`renewables` combines a flat-ish wind contribution with the solar hump, and `residual_load`
inherits both: highest in winter, deeply suppressed in the May–August midday season, and shifting
downward year on year as installed renewable capacity grows.

### 3.4 — Weekly rhythm

In [ ]:
PROFILE_COLS = ["grid_load", "residual_load", "renewables"]

dow_profile = time_series.groupby("dow")[PROFILE_COLS].mean()
dow_profile.index = DAY_NAMES

fig, ax = plt.subplots(figsize=(11, 5))
for col in PROFILE_COLS:
    ax.plot(dow_profile.index, dow_profile[col].to_numpy(), marker="o", **series_style(col))
ax.set_title("Mean level by day of week", fontsize=15, pad=12)
ax.set_ylabel("average MW", color="grey")
ax.set_xlabel("day of week")
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

weekend_contrast = time_series.groupby("is_weekend")[PROFILE_COLS].mean().T
weekend_contrast.columns = ["weekday", "weekend"]
weekend_contrast["delta_%"] = (
    100 * (weekend_contrast["weekend"] / weekend_contrast["weekday"] - 1)
).round(1)
display(weekend_contrast.round(0))

Monday to Friday form a flat plateau, Saturday drops and Sunday drops further — the classic demand
week. `renewables` is essentially flat across the week, as it must be: the weather does not know
what day it is. Because demand falls and supply does not, the weekend cut in `residual_load` is
**larger in percentage terms** than the one in `grid_load` — the weekly cycle is a residual-load
effect amplified by an unchanged renewable infeed.

### 3.5 — Daily rhythm, per season and day type

In [ ]:
hour_profile = time_series.groupby(["season", "is_weekend", "hour"], observed=True)["grid_load"].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True, sharex=True)

for ax, season in zip(axes.flat, SEASON_ORDER):
    for weekend, label in [(False, "weekday"), (True, "weekend")]:
        profile = hour_profile.loc[(season, weekend)]
        ax.plot(
            profile.index, profile.to_numpy(),
            marker="o", markersize=3, color=DAY_TYPE_COLOR[label], label=label,
        )
    ax.set_title(season, fontsize=13, pad=8)
    ax.set_xlabel("hour of day (Europe/Berlin)")
    ax.set_ylabel("average MW", color="grey")
    ax.set_xticks(range(0, 24, 3))
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
    ax.legend(frameon=False, fontsize=9)

fig.suptitle("Mean grid load by hour of day, per season and weekday/weekend", fontsize=16)
plt.tight_layout()
plt.show()

The daily shape is the same everywhere — overnight trough, steep 05:00–08:00 ramp, working-day
plateau, evening fall — but its **profile through the middle of the day changes with the season**.
In winter the weekday curve holds a broad plateau from 09:00 to 12:00 and then returns to a second
peak at 18:00 that is just as high, as darkness and heating meet the tail of the working day. In
summer that second peak is much weaker and the midday hours sag: the 12:00–16:00 stretch falls
below the morning maximum, because behind-the-meter solar serves load that therefore never appears
as grid load at all. Spring sits between the two; autumn resembles winter with a smaller
amplitude.

The weekday/weekend gap is a near-constant vertical offset in every season, with the **same
shape**: weekends keep the morning ramp and evening peak, simply lower and starting later. That is
a useful modelling fact — the weekend effect is close to multiplicative on the level rather than a
different daily shape.

### 3.6 — Federal holidays

German **federal** holidays only — `holidays.country_holidays("DE")` with no `subdiv`, the
project's single holiday source of truth. Grid load is a national quantity, and state-specific
holidays would make the comparison depend on which Bundesland happened to be picked. Each holiday
is compared against non-holiday days of the **same weekday** within ±14 days.

In [ ]:
holiday_years = range(min(YEARS), max(YEARS) + 1)  # from YEARS, never a literal
de_holidays = holidays.country_holidays("DE", years=holiday_years)  # no subdiv -> federal only
holiday_dates = set(de_holidays)  # set(), not the dict: Series.isin on a dict subclass is no contract

daily_load = time_series.groupby("date")[["grid_load", "residual_load"]].mean()
daily_load.index = pd.DatetimeIndex(daily_load.index)
daily_load["dow"] = daily_load.index.dayofweek
daily_load["is_holiday"] = pd.Series(daily_load.index.date, index=daily_load.index).isin(holiday_dates)

# State-only holidays would mean the call picked up a subdiv default; both spellings checked.
STATE_ONLY = {
    "Fronleichnam", "Allerheiligen", "Reformationstag", "Mariä Himmelfahrt",
    "Heilige Drei Könige", "Friedensfest", "Buß- und Bettag", "Weltkindertag",
    "Corpus Christi", "All Saints' Day", "Reformation Day", "Assumption Day",
    "Epiphany", "Peace Festival", "Repentance and Prayer Day", "World Children's Day",
}
holiday_names = {name for name in de_holidays.values()}
assert not (holiday_names & STATE_ONLY), sorted(holiday_names & STATE_ONLY)

print(
    f"{len(de_holidays)} federal holidays across {min(YEARS)}-{max(YEARS)}, "
    f"{int(daily_load['is_holiday'].sum())} of them inside the record"
)
print(f"distinct holiday names ({len(holiday_names)}): {sorted(holiday_names)}")

In [ ]:
rows = []
for day in daily_load.index[daily_load["is_holiday"]]:
    window = daily_load.loc[day - pd.Timedelta("14D"): day + pd.Timedelta("14D")]
    reference = window[(~window["is_holiday"]) & (window["dow"] == daily_load.at[day, "dow"])]
    if reference.empty:
        continue
    rows.append(
        {
            "holiday": de_holidays.get(day.date()),
            "date": day.date(),
            "weekday": day.day_name(),
            "load": daily_load.at[day, "grid_load"],
            "reference": reference["grid_load"].mean(),
            "delta_%": 100 * (daily_load.at[day, "grid_load"] / reference["grid_load"].mean() - 1),
        }
    )

holiday_effect = pd.DataFrame(rows)

by_name = (
    holiday_effect.groupby("holiday")
    .agg(n=("delta_%", "size"), mean_delta_pct=("delta_%", "mean"))
    .sort_values("mean_delta_pct")
    .round(1)
)
display(by_name)

**Every federal holiday depresses grid load** against comparable same-weekday days in the
surrounding four weeks, and the striking thing is how narrow the range is: no holiday behaves
qualitatively differently from the others. A holiday is worth roughly a fifth of a day's load,
whichever holiday it is — which makes a single "is holiday" feature a reasonable modelling
simplification rather than a lossy one.

That the list contains no Fronleichnam, Allerheiligen or Reformationstag is the check that the
no-`subdiv` call really did return federal holidays only; the assertion above fails loudly if a
state-level holiday ever appears.

**One caveat on method.** For the Christmas holidays the ±14-day reference window sits partly
inside the depressed Christmas period itself, so those deltas **understate** the true effect. The
numbers above are a lower bound for Christmas Day and Boxing Day, not a measurement of the full
holiday-season dip.

### 3.7 — Where the residual-load tails sit on the calendar

The extreme 1 % of hours at each end, selected **by rank**: this is a descriptive slice, not a
risk definition, and no flag column is written back onto `time_series`. Colours come from §1.2's
tail pair: blue for the low (oversupply) end, red for the high (tight-margin) end.

In [ ]:
n_tail = round(0.01 * len(time_series))
low_tail = time_series["residual_load"].nsmallest(n_tail)   # descriptive slice, never persisted
high_tail = time_series["residual_load"].nlargest(n_tail)

print(f"{n_tail:,} hours in each tail (1 % of the record by rank)")
print(f"  low  tail spans {low_tail.min():>10,.0f} .. {low_tail.max():>10,.0f} MW")
print(f"  high tail spans {high_tail.min():>10,.0f} .. {high_tail.max():>10,.0f} MW")

DIMENSIONS = [
    ("year", lambda idx: idx.year, None),
    ("month", lambda idx: idx.month, range(1, 13)),
    ("hour of day", lambda idx: idx.hour, range(24)),
    ("day of week", lambda idx: idx.dayofweek, range(7)),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

for ax, (label, extract, full_range) in zip(axes.flat, DIMENSIONS):
    frame = pd.DataFrame(
        {
            "lowest 1 % (oversupply)": pd.Series(extract(low_tail.index)).value_counts(normalize=True),
            "highest 1 % (tight margin)": pd.Series(extract(high_tail.index)).value_counts(normalize=True),
        }
    )
    if full_range is not None:
        frame = frame.reindex(list(full_range))
    frame = frame.sort_index() * 100

    x = np.arange(len(frame))
    ax.bar(x - 0.2, frame.iloc[:, 0].to_numpy(), width=0.4, label=frame.columns[0],
           color=TAIL_COLOR["low"])
    ax.bar(x + 0.2, frame.iloc[:, 1].to_numpy(), width=0.4, label=frame.columns[1],
           color=TAIL_COLOR["high"])
    ax.set_xticks(x)
    ax.set_xticklabels(
        DAY_NAMES if label == "day of week" else [str(v) for v in frame.index], fontsize=9
    )
    ax.set_title(f"by {label}", fontsize=12, pad=8)
    ax.set_ylabel("share of that tail (%)", color="grey")
    ax.set_xlabel(label)
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, fontsize=9)

fig.suptitle(
    "Where the two residual load tails sit on the calendar (1 % by rank each)", fontsize=16
)
plt.tight_layout()
plt.show()

The two tails have **almost mirror-image calendar signatures**, which is the most useful thing in
this section.

**The low tail (oversupply)** is overwhelmingly recent, concentrated in the high-solar months,
in the midday hours, and at weekends — the weekly demand minimum of §3.4 lining up with the daily
solar maximum of §3.5. **The high tail (tight margins) is the mirror image**: weekday-only, spread
far more evenly across years rather than trending, sitting in the winter months and peaking at the
evening hours with a secondary morning cluster.

That contrast is worth dwelling on. The low tail is **growing and seasonal**; the high tail is
**stable and structural**. One is a new phenomenon created by renewable build-out, the other the
classic winter stress case that has always been there. For the modelling spec this is a strong
hint that **the two risk cases may not be one problem** — they run on different mechanisms, in
opposite seasons and at opposite hours. Whether to treat them as one target or two is a modelling
decision; this notebook only records that the evidence points toward two.

### 3.8 — Two representative extreme episodes

Selected by a **stated, reproducible, rank-based rule** rather than by eye:

1. **The longest run of consecutive negative `residual_load` hours.** Ties broken by earliest
   start — which matters, because the runners-up are only an hour shorter.
2. **The highest 24-hour rolling mean.** The window is time-based (`rolling("24h")`), not
   row-based, so the DST gaps cannot let a "24-row" window span 25 wall-clock hours.
   `min_periods=24` stops a partial window at the start of the record from winning.

The selection code below carries no tag of its own; it is ported because the tagged plot after it
cannot run without it.

In [ ]:
negative = time_series["residual_load"] < 0

# --- rule 1: longest consecutive negative run, ties broken by earliest start
blocks = (negative != negative.shift()).cumsum()
runs = (
    pd.DataFrame(
        {
            "length": negative.groupby(blocks).size(),
            "is_negative": negative.groupby(blocks).first(),
            "start": time_series.index.to_series().groupby(blocks).min(),
        }
    )
    .query("is_negative")
    .sort_values(["length", "start"], ascending=[False, True])
)
print("longest negative runs (top 5), showing the tie the rule has to break:")
display(runs.head(5)[["start", "length"]].reset_index(drop=True))

run_start = runs.iloc[0]["start"]
run_end = run_start + pd.Timedelta(hours=int(runs.iloc[0]["length"]) - 1)

# --- rule 2: highest 24-hour rolling mean, on a TIME-based window
rolling_24h = time_series["residual_load"].rolling("24h", min_periods=24).mean()
peak_end = rolling_24h.idxmax()
peak_start = peak_end - pd.Timedelta("23h")
print(f"\nhighest 24 h rolling mean: {rolling_24h.max():,.0f} MW over {peak_start} .. {peak_end}")

In [ ]:
EPISODE_COLS = ["residual_load", "grid_load", "wind_on", "wind_off", "solar"]
episodes = [
    (f"Longest negative run: {int(runs.iloc[0]['length'])} consecutive hours", run_start, run_end),
    (f"Highest 24 h rolling mean: {rolling_24h.max():,.0f} MW", peak_start, peak_end),
]

for title, start, end in episodes:
    window = time_series.loc[start - pd.Timedelta("36h"): end + pd.Timedelta("36h"), EPISODE_COLS]

    fig, ax = plt.subplots(figsize=(15, 5.5))
    for col in EPISODE_COLS:
        ax.plot(window.index, window[col].to_numpy(), **series_style(col))
    ax.axhline(0, color="0.35", linewidth=0.9)
    ax.axvspan(start, end, color="0.88", zorder=0, label="selected episode")

    style_timeseries(
        ax,
        f"{title}\n{start:%Y-%m-%d %H:%M} to {end:%Y-%m-%d %H:%M} (+/- 36 h context)",
        "MW",
    )
    # style_timeseries assumes a multi-year axis; a few-day window needs day ticks.
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=(0, 6, 12, 18)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.set_xlabel("Europe/Berlin local time")
    ax.legend(frameon=False, ncol=6, fontsize=9)
    plt.tight_layout()
    plt.show()

The two episodes show the two mechanisms in their pure form, and their **time scales are very
different**. The longest negative run is only twelve hours: a single late-summer day where solar
peaks above 30 GW against a weekend-level demand, pushing `residual_load` to roughly −14 GW around
midday before it recovers overnight. The ±36 h context shows the neighbouring days dipping below
zero as well — the oversupply case is a *recurring midday* phenomenon, not a multi-day state.

The high 24-hour mean is the opposite: a full winter day where wind has collapsed to near zero and
solar contributes almost nothing, so `residual_load` tracks `grid_load` almost exactly at a level
above 60 GW for a whole day — a *Dunkelflaute*-shaped episode where conventional generation and
imports carry essentially the entire load. Oversupply arrives in hours; tight margins arrive in
days.

### 3.9 — Self-check

In [ ]:
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert not (holiday_names & STATE_ONLY), "state-only holiday in the federal list"
assert len(low_tail) == len(high_tail) == n_tail
assert low_tail.max() < high_tail.min(), "the two tails overlap"
assert run_end > run_start and peak_end > peak_start

print("univariate/time-structure self-check passed")
print(f"  {len(holiday_names)} distinct federal holidays, none state-only")
print(f"  {n_tail:,} hours per tail, ranges disjoint")
print(f"  no column added to time_series ({len(time_series.columns)} columns)")